# Quickstart — governance for *your* pipeline in 5 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crbazevedo/delegation-lab/blob/main/notebooks/00_quickstart_your_pipeline.ipynb)

You have a workflow where agents (or people, or tools) hand work to each
other. This notebook turns a log of what each step **got right** — before and
after review — into a governance read on the whole pipeline:

- **Per node:** raw competence (σ_raw), corrected quality (σ_corr), and the
  **masking index M\*** = σ_corr / σ_raw — how much a downstream reviewer is
  hiding a weak step.
- **Whole pipeline:** the capacity ceiling **C_op** (the best corrected quality
  the final steps can reach), whether you clear your quality target, the
  **bottleneck**, and a **risk ranking**.
- **What to do:** concrete, prioritized prescriptions — where to add review,
  which node to upgrade, or whether the target is infeasible as designed.

Every number is computed by the open [`minimal-oversight`](https://pypi.org/project/minimal-oversight/)
package — the reference implementation of *Minimum Sufficient Oversight*.

> The **same trace JSON** you build below also pastes into the visual
> [governance cockpit](https://crbazevedo.github.io/delegation-lab/app/widgets/cockpit.html) → *Import your pipeline* — if you'd rather
> drag sliders than read tables.

## 1. Install

In [ ]:
%pip install -q minimal-oversight

## 2. Paste your traces

One row per (task, node). `outcome` is whether that step was correct **before**
any review (1/0); `corrected` is whether it was correct **after** the downstream
reviewer/corrector ran. If a step has no corrector, set `corrected` = `outcome`.

Below is a tiny worked example — a 3-step content pipeline
(`intake → draft → review`). **Replace it with your own log.** Even a few dozen
tasks is enough to see masking.

In [ ]:
EVENTS = [
    # task_id, node_id, outcome (pre-correction), corrected (post-correction)
    {"task_id": "t1", "node_id": "intake",   "outcome": 1, "corrected": 1},
    {"task_id": "t1", "node_id": "draft",    "outcome": 0, "corrected": 1},
    {"task_id": "t1", "node_id": "review",   "outcome": 1, "corrected": 1},
    {"task_id": "t2", "node_id": "intake",   "outcome": 1, "corrected": 1},
    {"task_id": "t2", "node_id": "draft",    "outcome": 1, "corrected": 1},
    {"task_id": "t2", "node_id": "review",   "outcome": 1, "corrected": 1},
    {"task_id": "t3", "node_id": "intake",   "outcome": 1, "corrected": 1},
    {"task_id": "t3", "node_id": "draft",    "outcome": 0, "corrected": 1},
    {"task_id": "t3", "node_id": "review",   "outcome": 1, "corrected": 1},
    {"task_id": "t4", "node_id": "intake",   "outcome": 1, "corrected": 1},
    {"task_id": "t4", "node_id": "draft",    "outcome": 0, "corrected": 1},
    {"task_id": "t4", "node_id": "review",   "outcome": 0, "corrected": 1},
    {"task_id": "t5", "node_id": "intake",   "outcome": 1, "corrected": 1},
    {"task_id": "t5", "node_id": "draft",    "outcome": 1, "corrected": 1},
    {"task_id": "t5", "node_id": "review",   "outcome": 1, "corrected": 1},
    {"task_id": "t6", "node_id": "intake",   "outcome": 1, "corrected": 1},
    {"task_id": "t6", "node_id": "draft",    "outcome": 0, "corrected": 1},
    {"task_id": "t6", "node_id": "review",   "outcome": 1, "corrected": 1},
    {"task_id": "t7", "node_id": "intake",   "outcome": 1, "corrected": 1},
    {"task_id": "t7", "node_id": "draft",    "outcome": 0, "corrected": 1},
    {"task_id": "t7", "node_id": "review",   "outcome": 1, "corrected": 1},
    {"task_id": "t8", "node_id": "intake",   "outcome": 1, "corrected": 1},
    {"task_id": "t8", "node_id": "draft",    "outcome": 1, "corrected": 1},
    {"task_id": "t8", "node_id": "review",   "outcome": 0, "corrected": 1},
    {"task_id": "t9", "node_id": "intake",   "outcome": 0, "corrected": 0},
    {"task_id": "t9", "node_id": "draft",    "outcome": 0, "corrected": 1},
    {"task_id": "t9", "node_id": "review",   "outcome": 1, "corrected": 1},
    {"task_id": "t10", "node_id": "intake",   "outcome": 0, "corrected": 0},
    {"task_id": "t10", "node_id": "draft",    "outcome": 1, "corrected": 1},
    {"task_id": "t10", "node_id": "review",   "outcome": 1, "corrected": 1},
]

## 3. Estimate each step from your traces

`from_generic_events` parses the log; `estimate_node` computes σ_raw, σ_corr,
the catch rate, and the masking index per node — straight from your data.

In [ ]:
from minimal_oversight.connectors.traces import from_generic_events, to_workflow_traces
from minimal_oversight.estimation import estimate_node

traces = to_workflow_traces(from_generic_events(EVENTS, corrected_field="corrected"))

# node order + parents, inferred from the order steps appear within each task
order, parents = [], {}
for tr in traces:
    prev = None
    for nid in tr.routing_path:
        if nid not in parents:
            parents[nid] = set(); order.append(nid)
        if prev:
            parents[nid].add(prev)
        prev = nid

est = {nid: estimate_node(nid, traces) for nid in order}

print(f"{len(traces)} tasks · {len(order)} nodes\n")
print(f"{'node':<10}{'sigma_raw':>10}{'sigma_corr':>11}{'catch':>8}{'M*':>7}")
for nid in order:
    e = est[nid]
    c = '-' if e.catch_rate is None else f'{e.catch_rate:.2f}'
    print(f"{nid:<10}{e.sigma_raw:>10.3f}{e.sigma_corr:>11.3f}{c:>8}{e.masking_index:>7.2f}")

**Reading it:** M\* > 1 means the reviewer is doing real work — and hiding how
weak the raw step is. A step with high M\* looks fine in your output while
quietly depending on review. That is exactly where a silent model regression
would slip through unnoticed.

## 4. Read the governance off the whole pipeline

Build the graph from the estimates and run the full analysis: the capacity
ceiling C_op, feasibility against your target, the bottleneck, and a risk
ranking by how far each node's errors propagate downstream.

We set `sigma_skill = σ_raw / γ` (γ = 10/12, the return-operator fixed-point
gain) alongside the estimated σ_raw/σ_corr, so the model reproduces your
estimates *and* the recommender can reason about each node.

In [ ]:
from minimal_oversight.models import Node, PipelineGraph
from minimal_oversight import analyze_pipeline

P_MIN = 0.80     # your quality target (corrected success rate you need to ship)
GAMMA = 10 / 12  # return-operator fixed-point gain: sigma_raw* = gamma * sigma_skill
clamp = lambda x, lo, hi: max(lo, min(hi, x))

def make_graph():
    nodes = []
    for nid in order:
        e = est[nid]
        nodes.append(Node(
            nid,
            sigma_skill=clamp(e.sigma_raw / GAMMA, 0.05, 0.98),
            sigma_raw=e.sigma_raw,
            sigma_corr=e.sigma_corr,
            catch_rate=0.6 if e.catch_rate is None else e.catch_rate,
        ))
    g = PipelineGraph(nodes)
    for nid in order:
        for p in parents[nid]:
            g.add_edge(p, nid)
    return g

report = analyze_pipeline(make_graph(), p_min=P_MIN)
f = report.feasibility
verdict = 'FEASIBLE' if f.feasible else 'INFEASIBLE'
print(f"{verdict}  ·  C_op={f.c_op:.3f}  ·  B_eff={f.b_eff:+.3f}  ·  target p_min={P_MIN}")
print(f"bottleneck: {f.bottleneck_node}\n")

print(f"{'node':<10}{'M*':>7}{'centrality':>12}{'risk (S)':>10}")
for r in report.node_risks:
    s = '-' if r.sota_score is None else f'{r.sota_score:.2f}'
    print(f"{r.name:<10}{r.masking_index:>7.2f}{r.delegation_centrality:>12.2f}{s:>10}")

Notice the twist: this pipeline is **feasible** — C_op clears the target — yet
`draft` carries a high masking index at high centrality. The headline looks
healthy while the real risk hides one layer down. That gap is the whole point of
MSO, and it's why the next cell still has work to do.

## 5. What to do about it

`recommend_governance_changes` combines capacity, topology, and masking into an
ordered, node-level action list — not just diagnostics.

In [ ]:
from minimal_oversight import recommend_governance_changes

recs = recommend_governance_changes(make_graph(), p_min=P_MIN)
actionable = [r for r in recs if not r.action.startswith('Monitor')]
if not actionable:
    print('No changes needed — every node clears the bar with headroom.')
for rec in recs:
    print(f"[P{rec.priority}] {rec.action}"
          + (f"  ·  node: {rec.target_node}" if rec.target_node else ''))
    print(f"       why: {rec.rationale.splitlines()[0]}")
    print(f"       impact: {rec.expected_impact}\n")

## See it visually — and run it on your system

Open the [governance cockpit](https://crbazevedo.github.io/delegation-lab/app/widgets/cockpit.html), click **Import your pipeline**, and
paste the JSON printed by the cell below. You'll get the live graph with sliders
— move a reviewer upstream, raise a node's competence, add a blind audit — and
watch C_op and masking respond in real time.

In [ ]:
import json
print(json.dumps(EVENTS))  # copy this line's output into the cockpit's import box


**To run this on your real system, you only need to log, per task and per node:**

1. `outcome` — was this step correct *before* anything downstream reviewed it?
2. `corrected` — was it correct *after* its reviewer/corrector ran?

That's the whole data contract. With it you get masking, bottlenecks, and a
prioritized fix list — the minimum sufficient oversight for your pipeline.

Already on a framework? See [`04_langgraph_import`](04_langgraph_import.ipynb)
(import a LangGraph `StateGraph` directly) and
[`05_adk_import`](05_adk_import.ipynb).